===========================================================================
# Generative AI Healthcare Translator - User Interface
===========================================================================

This notebook provides a simple user-facing interface for querying the healthcare narrative data developed in the project.

### What the user does
1. Select a healthcare domain.
2. Type a natural-language question.
3. Choose how many supporting records to retrieve.
4. Ask the project
5. Read the generated answer and inspect the supporting evidence.

The interface is designed to reuse the project's existing dataframes, embedding model, and generation pipeline whenever they are already loaded.


## Task 1: Imports and Setup
--------------------------------------------------------------------------


In [17]:
# Import libraries

import warnings
warnings.filterwarnings("ignore")

import html
import json
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

print("Interface dependencies loaded.")

Interface dependencies loaded.


In [18]:
# Run the application notebook

%run "6 - Application.ipynb"

Evidence Rows: 57298
Prompt Experiment Rows: 60
Testing Results: 20
Communication Styles: 4

Communication Styles:
['Patient Friendly', 'Executive Summary', 'Clinical', 'Community Report']

Testing Results by Audience:


,Audience,Narratives
0,Clinical,5
1,Community Report,5
2,Executive Summary,5
3,Patient Friendly,5


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation Model: google/flan-t5-large
APPLICATION RESULT
Facility: MONUMENT HEALTH RAPID CITY HOSPITAL
Communication Style: Patient Friendly
Prompt Version: 1

PERFORMANCE EVIDENCE
--------------------------------------------------------------------------------
Facility: MONUMENT HEALTH RAPID CITY HOSPITAL

Strengths:
  Timely and Effective Care
  Relative Performance: 0.45 SD from cohort

  Key Measures:
    - ED Time - Psychiatric Patients: 179.00 Minutes
      Cohort Mean: 319.04 | Lower is Better
    - ED Time - All Patients: 162.00 Minutes
      Cohort Mean: 196.20 | Lower is Better

Weaknesses:
  Patient Survey

  Key Measures:
    - Recommend hospital: 3.00 Stars
      Cohort Mean: 3.41 | Higher is Better
    - Communication about medicines: 2.00 Stars
      Cohort Mean: 2.25 | Higher is Better

CONTEXT EVIDENCE
--------------------------------------------------------------------------------
nan

GENERATED NARRATIVE
--------------------------------------------------------------

## Task 2: Connect Project Data
--------------------------------------------------------------------------


In [19]:
# Task 2: Locate and connect project data

search_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent
]

def find_project_file(filename):

    for root in search_roots:
        matches = list(root.rglob(filename))

        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not locate {filename} from the available project folders.")


evidence_file = find_project_file("evidence_repository.csv")
prompt_experiment_file = find_project_file("prompt_experiment.csv")
communication_prompts_file = find_project_file("communication_prompts.json")

print("FILES LOCATED")
print("=" * 80)
print("Evidence Repository:", evidence_file)
print("Prompt Experiment:", prompt_experiment_file)
print("Communication Prompts:", communication_prompts_file)

evidence_repository = pd.read_csv(evidence_file)
prompt_experiment = pd.read_csv(prompt_experiment_file)

with open(communication_prompts_file, "r", encoding="utf-8") as file:
    communication_prompts = json.load(file)

print("\nPROJECT DATA LOADED")
print("=" * 80)
print("Evidence Rows:", len(evidence_repository))
print("Prompt Experiment Rows:", len(prompt_experiment))
print("Communication Styles:", list(communication_prompts.keys()))

FILES LOCATED
Evidence Repository: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Final Project\Data\Narratives\evidence_repository.csv
Prompt Experiment: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Final Project\Data\Narratives\prompt_experiment.csv
Communication Prompts: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Final Project\Data\Narratives\communication_prompts.json

PROJECT DATA LOADED
Evidence Rows: 57298
Prompt Experiment Rows: 60
Communication Styles: ['Patient Friendly', 'Executive Summary', 'Clinical', 'Community Report']


In [20]:
# Task 2: Connect project data

narrative_data_dir = Path("Data/Narratives")
evaluation_data_dir = Path("Data/Evaluation")

evidence_file = narrative_data_dir / "evidence_repository.csv"
prompt_experiment_file = narrative_data_dir / "prompt_experiment.csv"
communication_prompts_file = narrative_data_dir / "communication_prompts.json"


evidence_repository = pd.read_csv(evidence_file)
prompt_experiment = pd.read_csv(prompt_experiment_file)

with open(communication_prompts_file, "r", encoding="utf-8") as file:
    communication_prompts = json.load(file)

print("Evidence Rows:", len(evidence_repository))
print("Prompt Experiment Rows:", len(prompt_experiment))
print("Communication Styles:", list(communication_prompts.keys()))

Evidence Rows: 57298
Prompt Experiment Rows: 60
Communication Styles: ['Patient Friendly', 'Executive Summary', 'Clinical', 'Community Report']


In [28]:
# Task 3: Create user interface controls

available_facilities = sorted(prompt_experiment["Facility Name"].dropna().unique())
available_styles = list(communication_prompts.keys())

facility_dropdown = widgets.Dropdown(
    options=available_facilities,
    description="Hospital:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="95%")
)

style_dropdown = widgets.Dropdown(
    options=available_styles,
    value="Patient Friendly",
    description="Style:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="95%")
)

generate_button = widgets.Button(
    description="Generate Narrative",
    button_style="primary"
)

clear_button = widgets.Button(description="Clear")

status_html = widgets.HTML()
output_area = widgets.Output()

In [29]:
# Task 3: Retrieve prepared prompt for selected hospital and communication style

def retrieve_prepared_prompt(facility_id, communication_style="Patient Friendly", prompt_version=1):

    prepared = prompt_experiment[
        (prompt_experiment["Facility ID"].astype(str) == str(facility_id)) &
        (prompt_experiment["Audience"] == communication_style) &
        (prompt_experiment["Prompt Version"] == prompt_version)
    ].copy()

    if prepared.empty:
        return None

    return prepared.iloc[0]

In [30]:
# Task 3: Load generation model

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

generation_model_name = "google/flan-t5-large"

generation_tokenizer = AutoTokenizer.from_pretrained(generation_model_name)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(generation_model_name)
generation_model.eval()

generation_parameters = {
    "max_new_tokens": 200,
    "min_new_tokens": 40,
    "do_sample": False,
    "num_beams": 2,
    "no_repeat_ngram_size": 2,
    "early_stopping": True
}

print("Generation Model:", generation_model_name)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation Model: google/flan-t5-large


In [31]:
# Create narrative generation function

def generate_text(prompt, max_new_tokens=200, min_new_tokens=40, do_sample=False,
                  num_beams=2, no_repeat_ngram_size=2, early_stopping=True):

    inputs = generation_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=do_sample,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=early_stopping
        )

    return generation_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [32]:
# Task 4: Generate hospital communication

def hospital_communication_app(facility_name=None, facility_id=None,
                               communication_style="Patient Friendly", prompt_version=1):

    if communication_style not in communication_prompts:
        raise ValueError(f"Communication style must be one of: {list(communication_prompts.keys())}")

    if facility_id is None:
        facility_match = evidence_repository[
            evidence_repository["Facility Name"].str.lower() == str(facility_name).lower()
        ].copy()

        if facility_match.empty:
            return {
                "Facility Name": facility_name,
                "Communication Style": communication_style,
                "Prompt Version": prompt_version,
                "Performance Evidence": None,
                "Context Evidence": None,
                "Generated Narrative": "No matching hospital was found."
            }

        facility_id = facility_match["Facility ID"].iloc[0]

    prepared = retrieve_prepared_prompt(
        facility_id=facility_id,
        communication_style=communication_style,
        prompt_version=prompt_version
    )

    if prepared is None:
        return {
            "Facility Name": facility_name,
            "Communication Style": communication_style,
            "Prompt Version": prompt_version,
            "Performance Evidence": None,
            "Context Evidence": None,
            "Generated Narrative": "No prepared prompt was found for this hospital and style."
        }

    generated_narrative = generate_text(prepared["Prompt"], **generation_parameters)

    return {
        "Facility ID": facility_id,
        "Facility Name": prepared["Facility Name"],
        "Communication Style": communication_style,
        "Prompt Version": prompt_version,
        "Performance Evidence": prepared["Performance Evidence"],
        "Context Evidence": prepared["Context Evidence"],
        "Prompt": prepared["Prompt"],
        "Generated Narrative": generated_narrative
    }

In [33]:
# Task 4: Connect interface to hospital communication application

def handle_generate(_):

    output_area.clear_output()

    facility_name = facility_dropdown.value
    communication_style = style_dropdown.value

    status_html.value = "<i>Generating narrative...</i>"
    generate_button.disabled = True

    try:
        result = hospital_communication_app(
            facility_name=facility_name,
            communication_style=communication_style,
            prompt_version=1
        )

        with output_area:
            print("HOSPITAL PERFORMANCE COMMUNICATION")
            print("=" * 90)
            print("Hospital:", facility_name)
            print("Communication Style:", result["Communication Style"])
            print("Prompt Version:", result["Prompt Version"])

            print("\nPERFORMANCE EVIDENCE")
            print("-" * 90)
            print(result["Performance Evidence"])

            if result["Context Evidence"]:
                print("\nCONTEXT EVIDENCE")
                print("-" * 90)
                print(result["Context Evidence"])

            print("\nGENERATED NARRATIVE")
            print("-" * 90)
            print(result["Generated Narrative"])

        status_html.value = "<b>Generation complete.</b>"

    except Exception as exc:
        message = html.escape(f"{type(exc).__name__}: {exc}")
        status_html.value = f"<b style='color:#b42318;'>Generation failed.</b><br><code>{message}</code>"

    finally:
        generate_button.disabled = False


def handle_clear(_):
    status_html.value = ""
    output_area.clear_output()


generate_button.on_click(handle_generate)
clear_button.on_click(handle_clear)

In [34]:
# Task 5: Display final user interface

header = widgets.HTML(
    """
    <h2>Hospital Performance Communication Assistant</h2>
    <p>
    Select a hospital and communication style to generate an audience-specific
    hospital performance narrative from the project's prepared evidence.
    </p>
    """
)

controls = widgets.VBox([
    facility_dropdown,
    style_dropdown,
    widgets.HBox([generate_button, clear_button]),
    status_html
])

interface = widgets.VBox([
    header,
    controls,
    widgets.HTML("<hr>"),
    output_area
])

display(interface)